**Data required:** Run `04_homophily_simulations/run_simulations.py` first to populate `04_homophily_simulations/output/` with simulation pkl files. These baseline simulations do not back a figure in the revised manuscript; the swap and finite-N analyses that do are in 06–08.

In [ ]:
import pickle
import numpy as np
import pandas as pd

In [ ]:

from pyprojroot import here

import importlib.util

_spec = importlib.util.spec_from_file_location("opinion_functions", here() / "src" / "opinion_functions.py")
_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
fun = _mod



In [ ]:
# Parameters
input_folder = here() / "04_homophily_simulations" / "output"
homophilyvec = [0, 0.25, 0.5, 0.75, 1]
m_values = [2, 5]
num_agents_values = [1000]
num_sim = 1000

simulation_results = fun.load_simulation_results(input_folder, homophilyvec, m_values, num_agents_values, num_sim)

In [ ]:
## Specify simulation parameters to compute summary statistics for
m = 5
num_agents = 1000
minority_fraction = 0.333
majority_fraction = 1 - minority_fraction

In [ ]:
# Initialize dictionaries to store results and their statistics for each key
mean_majority_opinion_stats = {}
mean_minority_opinion_stats = {}
mean_overall_opinion_stats = {}

# Iterate over homophily values
for i, homophily in enumerate(homophilyvec):
    result_key = (homophily, m, num_agents)
    if result_key not in simulation_results:
        print(f"No data for homophily={homophily}, m={m}, num_agents={num_agents}")
        continue

    # Extract simulation data
    majority_data = simulation_results[result_key]["majority"]
    minority_data = simulation_results[result_key]["minority"]

    # Compute mean opinions across simulations
    mean_majority = np.mean(majority_data, axis=1)  # Mean for each simulation
    mean_minority = np.mean(minority_data, axis=1)  # Mean for each simulation
    mean_overall = (
        minority_fraction * mean_minority + majority_fraction * mean_majority
    )

    # Compute statistics for majority opinion
    mean_majority_mean = np.mean(mean_majority)
    mean_majority_std = np.std(mean_majority)
    mean_majority_min = np.min(mean_majority)
    mean_majority_max = np.max(mean_majority)

    # Compute statistics for minority opinion
    mean_minority_mean = np.mean(mean_minority)
    mean_minority_std = np.std(mean_minority)
    mean_minority_min = np.min(mean_minority)
    mean_minority_max = np.max(mean_minority)

    # Compute statistics for overall opinion
    mean_overall_mean = np.mean(mean_overall)
    mean_overall_std = np.std(mean_overall)
    mean_overall_min = np.min(mean_overall)
    mean_overall_max = np.max(mean_overall)

    # Store the results and their statistics in dictionaries
    mean_majority_opinion_stats[result_key] = {
        "array": mean_majority,
        "mean": mean_majority_mean,
        "std": mean_majority_std,
        "min": mean_majority_min,
        "max": mean_majority_max,
    }
    mean_minority_opinion_stats[result_key] = {
        "array": mean_minority,
        "mean": mean_minority_mean,
        "std": mean_minority_std,
        "min": mean_minority_min,
        "max": mean_minority_max,
    }
    mean_overall_opinion_stats[result_key] = {
        "array": mean_overall,
        "mean": mean_overall_mean,
        "std": mean_overall_std,
        "min": mean_overall_min,
        "max": mean_overall_max,
    }


In [ ]:
# Initialize a dictionary to store the data
multiindex_data = []

# Helper function to add rows for each result_key
def add_to_multiindex_data(result_key, stats_dicts):
    row = {
        "homophily": result_key[0],
    }
    # Add statistics for each opinion type
    for opinion_type, stats_dict in stats_dicts.items():
        row[(opinion_type, "mean")] = np.round(stats_dict["mean"], 2)
        row[(opinion_type, "std")] = np.round(stats_dict["std"], 2)
        row[(opinion_type, "min")] = np.round(stats_dict["min"], 2)
        row[(opinion_type, "max")] = np.round(stats_dict["max"], 2)
    multiindex_data.append(row)

# Combine the statistics into rows
m_value = None
num_agents_value = None
for result_key in mean_majority_opinion_stats.keys():
    homophily, m_value, num_agents_value = result_key
    stats_dicts = {
        "majority": mean_majority_opinion_stats[result_key],
        "minority": mean_minority_opinion_stats[result_key],
        "overall": mean_overall_opinion_stats[result_key],
    }
    add_to_multiindex_data(result_key, stats_dicts)

# Create the DataFrame
multiindex_df = pd.DataFrame(multiindex_data)

# Set the MultiIndex columns
multiindex_df = multiindex_df.set_index(["homophily"])
multiindex_df.columns = pd.MultiIndex.from_tuples(multiindex_df.columns)

# Construct the file name
file_name = here() / "04_homophily_simulations" / "summary_stats" / f"summary_statistics_m{m_value}_agents{num_agents_value}.csv"

# Save the DataFrame to the file
multiindex_df.to_csv(file_name)

print(f"DataFrame saved to {file_name}")

In [ ]:
multiindex_df